In [1]:

library(MultiAssayExperiment)
library(RaggedExperiment)
library(data.table)


Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges

In [2]:
drop_sa <- fread("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv")
drop_sa[, master_pid := gsub( "_", ".", nct_pid)]

In [3]:
load("/omics/odcf/analysis/OE0246_projects/datamaster/dataMASTER_release/object/dataMASTER_241104123708.RData")
load("/omics/odcf/analysis/OE0246_projects/datamaster/ext_data/annotations/gencode19_gns_lite.RData")


In [4]:
myobj = dataMASTER[,,c("cnv", "cnv_germline", "sv")] # class: MAE




Warning message:
“'experiments' dropped; see 'drops()'”


harmonizing input:
  removing 62967 sampleMap rows not in names(experiments)
  removing 105 colData rownames not in sampleMap 'primary'



In [5]:
# 1. Get all unique column names from the MEA
all_colnames <- unlist(colnames(myobj))

# 2. Initialize an empty character vector to store matches
matched_cols <- character()

# 3. Loop through each ID in your sa table
for (pid in drop_sa$master_pid) {
  # We use fixed = TRUE because your IDs have dots (.) 
  # which would otherwise be treated as wildcards in regex
  hits <- grep(pid, all_colnames, value = TRUE, fixed = TRUE)
  
  # Append new hits to our master list
  matched_cols <- c(matched_cols, hits)
}

# 4. Remove duplicates (in case one column matched multiple PIDs)
matched_cols <- unique(matched_cols)

In [13]:
germline_cnv_df <- as.data.frame(myobj[["cnv_germline"]])
library(data.table)
setDT(germline_cnv_df)

# Take a look at the first few rows
germline_cnv_df 

group,group_name,seqnames,start,end,width,strand,Type,CN,NP,QUAL,Confidence
<int>,<chr>,<fct>,<int>,<int>,<int>,<fct>,<fct>,<int>,<int>,<int>,<fct>
1,WES.192P9WW.tumor,3,11645548,11646305,758,*,DEL,1,1,65,LOW
1,WES.192P9WW.tumor,6,123698950,123699293,344,*,DEL,1,1,42,HIGH
1,WES.192P9WW.tumor,16,70993300,70993973,674,*,DEL,0,1,44,HIGH
1,WES.192P9WW.tumor,19,14877540,14878171,632,*,DEL,1,1,34,LOW
1,WES.192P9WW.tumor,19,15784117,15789465,5349,*,DEL,0,2,645,HIGH
1,WES.192P9WW.tumor,22,17264052,17265549,1498,*,DEL,0,1,163,HIGH
1,WES.192P9WW.tumor,X,153418163,153418831,669,*,DEL,0,1,205,HIGH
1,WES.192P9WW.tumor,Y,14951540,14959121,7582,*,DUP,3,6,51,LOW
2,WES.8ZTUHJ.metastasis,1,151336530,151403567,67038,*,DUP,3,35,145,HIGH


In [6]:
cnv_germline  = getWithColData(myobj, "cnv_germline")
actual_cnv_cols <- colnames(cnv_germline)

# 2. Filter your matched_cols to only include those present in CNV
valid_cnv_matches <- intersect(matched_cols, actual_cnv_cols)

# 3. Now subset safely
cnv_germline <- cnv_germline[, valid_cnv_matches, ]
cnv_germline


cnv_germline@assays[[1]]$Gene = as.character(NA)
sv = as(cnv_germline, "GRangesList")
 
n = length(sv)
sv = lapply(1:100, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["cnv_germline"]] = sv_re


germline_cnv_dt <- as.data.table(myobj[["cnv_germline"]]@assays[[1]])
#fwrite(germline_cnv_dt, "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/germline_cnv.tsv", sep="\t")

class: RaggedExperiment 
dim: 100103 3817 
assays(5): Type CN NP QUAL Confidence
rownames: NULL
colnames(3817): WES.00JN.metastasis WES.019E.metastasis ...
  WGS.ZZU2PY.tumor WGS.ZZXYWB.metastasis05
colData names(59): SampleID ProjectID ... INDEL_ClinFunOnTarget
  MutPerMb

ERROR: Error in RaggedExperiment(sv, colData = colData(cnv_germline)): length() of 'rowRanges' is not equal to nrow() of 'colData'


seqnames,start,end,width,strand,Type,CN,NP,QUAL,Confidence,Gene
<fct>,<int>,<int>,<int>,<fct>,<fct>,<int>,<int>,<int>,<fct>,<chr>
6,32606437,32611248,4812,*,DEL,1,6,112,LOW,HLA-DQA1
7,100550616,100551693,1078,*,DUP,3,4,31,HIGH,MUC3A
7,122254929,122262012,7084,*,DEL,1,2,118,HIGH,CADPS2
10,124344469,124351722,7254,*,DEL,1,10,531,HIGH,DMBT1
11,18955479,18956604,1126,*,DUP,3,4,30,LOW,"MRGPRX1,RP11-583F24.8"
12,131953679,131954299,621,*,DEL,0,1,58,LOW,RP11-897M7.1
12,133779224,133811272,32049,*,DUP,3,15,126,HIGH,"ZNF268,AC226150.4,AC226150.3,ANHX"
14,106372747,106376611,3865,*,DEL,1,4,147,LOW,"IGHD2-8,IGHD1-7,IGHD6-6"
17,19532621,19536854,4234,*,DEL,1,4,139,HIGH,CTD-2104P17.1


In [ ]:
cnv  = getWithColData(myobj, "cnv")
actual_cnv_cols <- colnames(cnv)

# 2. Filter your matched_cols to only include those present in CNV
valid_cnv_matches <- intersect(matched_cols, actual_cnv_cols)

# 3. Now subset safely
cnv <- cnv[, valid_cnv_matches, ]
cnv


cnv@assays[[1]]$Gene = as.character(NA)
sv = as(cnv, "GRangesList")
 
n = length(sv)
sv = lapply(1:n, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["cnv"]] = sv_re

# show the example
head(myobj[["cnv"]]@assays[[1]])

In [ ]:
head(myobj[["cnv"]]@assays[[1]])

In [5]:
cnv_germline  = getWithColData(myobj, "cnv_germline")
assayNames(cnv_germline)
cnv_germline = cnv_germline[, "WGS.MNL8UP.tumor"]
cnv_germline



[1] "Type"       "CN"         "NP"         "QUAL"       "Confidence"

class: RaggedExperiment 
dim: 100103 1 
assays(5): Type CN NP QUAL Confidence
rownames: NULL
colnames(1): WGS.MNL8UP.tumor
colData names(59): SampleID ProjectID ... INDEL_ClinFunOnTarget
  MutPerMb

In [ ]:
cnv_germline@assays[[1]]$Gene = as.character(NA)
sv = as(cnv_germline, "GRangesList")
sv

GRangesList object of length 1:
$WGS.MNL8UP.tumor
GRanges object with 4 ranges and 6 metadata columns:
      seqnames            ranges strand |     Type        CN        NP
         <Rle>         <IRanges>  <Rle> | <factor> <integer> <integer>
  [1]        2 27872732-28211204      * |      DUP         3        45
  [2]        3 75758544-75788746      * |      DEL         0         3
  [3]       19 15784190-15789625      * |      DEL         0         2
  [4]       22 24375883-24384930      * |      DUP         4         7
           QUAL Confidence        Gene
      <integer>   <factor> <character>
  [1]      1123       HIGH        <NA>
  [2]       425       LOW         <NA>
  [3]       224       LOW         <NA>
  [4]        73       LOW         <NA>
  -------
  seqinfo: 24 sequences from hg19_1KGRef_PhiX genome


In [6]:
cnv_germline@assays[[1]]$Gene = as.character(NA)
sv = as(cnv_germline, "GRangesList")
 
n = length(sv)
sv = lapply(1:n, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["cnv_germline"]] = sv_re

# show the example
head(myobj[["cnv_germline"]]@assays[[1]])

harmonizing input:
  removing 5613 sampleMap rows with 'colname' not in colnames of experiments



GRanges object with 4 ranges and 6 metadata columns:
      seqnames            ranges strand |     Type        CN        NP
         <Rle>         <IRanges>  <Rle> | <factor> <integer> <integer>
  [1]        2 27872732-28211204      * |      DUP         3        45
  [2]        3 75758544-75788746      * |      DEL         0         3
  [3]       19 15784190-15789625      * |      DEL         0         2
  [4]       22 24375883-24384930      * |      DUP         4         7
           QUAL Confidence                   Gene
      <integer>   <factor>            <character>
  [1]      1123       HIGH GPN1,SUPT7L,SLC4A1AP..
  [2]       425       LOW  ZNF717,RN7SL92P,MIR4..
  [3]       224       LOW                 CYP4F12
  [4]        73       LOW        KB-226F1.1,GSTT1
  -------
  seqinfo: 24 sequences from hg19_1KGRef_PhiX genome

In [8]:
cnv_germline_dt <- as.data.table(myobj[["cnv_germline"]]@assays[[1]])
cnv_germline_dt

seqnames,start,end,width,strand,Type,CN,NP,QUAL,Confidence,Gene
<fct>,<int>,<int>,<int>,<fct>,<fct>,<int>,<int>,<int>,<fct>,<chr>
2,27872732,28211204,338473,*,DUP,3,45,1123,HIGH,"GPN1,SUPT7L,SLC4A1AP,AC074091.13,MRPL33,AC110084.1,RBKS,BRE,RP11-674I16.1"
3,75758544,75788746,30203,*,DEL,0,3,425,LOW,"ZNF717,RN7SL92P,MIR4273"
19,15784190,15789625,5436,*,DEL,0,2,224,LOW,CYP4F12
22,24375883,24384930,9048,*,DUP,4,7,73,LOW,"KB-226F1.1,GSTT1"


In [17]:
cnv@assays[[1]]$Gene = as.character(NA)
sv = as(cnv, "GRangesList")
 
n = length(sv)
sv = lapply(1:n, function(i) {
  svPID = sv[[i]]
  # cat(sprintf("\r%s/%s", i, n))
  idx = findOverlaps(svPID, gencode19_gns_lite)
  gns = tapply(idx@to, idx@from, function(i) {
    paste(gencode19_gns_lite$gene_name[i], collapse=",")
  })
  svPID$Gene[as.numeric(names(gns))] = unname(gns)
  svPID
})
sv_re = RaggedExperiment(sv, colData=colData(cnv_germline))

myobj[["cnv"]] = sv_re

# show the example
head(myobj[["cnv"]]@assays[[1]])
cnv_dt <- as.data.table(myobj[["cnv"]]@assays[[1]])
cnv_dt


GRanges object with 6 ranges and 8 metadata columns:
      seqnames            ranges strand |       TCN  CovRatio       BAF
         <Rle>         <IRanges>  <Rle> | <numeric> <numeric> <numeric>
  [1]        1   840009-12839977      * |   2.00164   1.04347        NA
  [2]        1 13780016-19490045      * |   2.00164   1.04347        NA
  [3]        1 19490046-19499979      * |   1.82232   0.95840        NA
  [4]        1 19499980-25510000      * |   2.00164   1.04347        NA
  [5]        1 25510001-25520840      * |   1.80672   0.95100        NA
  [6]        1 25520841-51969987      * |   2.00164   1.04347        NA
         c1Mean    c2Mean     Type       LOH                   Gene
      <numeric> <numeric> <factor> <logical>            <character>
  [1]   1.00082   1.00082      CNN     FALSE RP11-54O7.16,RP11-54..
  [2]   1.00082   1.00082      CNN     FALSE LRRC38,RP4-597A16.2,..
  [3]        NA        NA      CNN     FALSE                   UBR4
  [4]   1.00082   1.00082      

seqnames start     end       width    strand TCN         CovRatio 
1   1           840009  12839977 11999969 *      2.001637    1.0434750
2   1         13780016  19490045  5710030 *      2.001637    1.0434750
3   1         19490046  19499979     9934 *      1.822315    0.9584000
4   1         19499980  25510000  6010021 *      2.001637    1.0434750
5   1         25510001  25520840    10840 *      1.806718    0.9510000
6   1         25520841  51969987 26449147 *      2.001637    1.0434750
7   1         51969987  51980013    10027 *      2.179377    1.1278000
8   1         51980014 120529963 68549950 *      2.001637    1.0434750
9   1        121220004 121229986     9983 *      2.001637    1.0434750
10  1        144010127 144019970     9844 *      2.039208    1.0613000
11  1        144920087 144948894    28808 *      2.001637    1.0434750
12  1        144948895 144952639     3745 *      2.073565    1.0776000
13  1        144952640 145739956   787317 *      2.001637    1.0434750
14  1        146540101 147829895  1289795 *      2.001637    1.0434750
15  1        149815060 220124938 70309879 *      2.001637    1.0434750
16  1        220124939 220133381     8443 *      1.348692    0.7337000
17  1        220133382 220223020    89639 *      1.067933    0.6005000
18  1        220223021 249209945 28986925 *      2.001637    1.0434750
19  2            20016     30021    10006 *      2.279287    1.1752000
20  2            30022  19119939 19089918 *      2.001637    1.0434750
21  2         19119939  19130003    10065 *      2.388050    1.2268000
22  2         19130004  33179511 14049508 *      2.001637    1.0434750
23  2         33179512  33460507   280996 *      1.002026    0.5692317
24  2         33460508  44385000 10924493 *      2.001637    1.0434750
25  2         44385001  44410022    25022 *      2.142912    1.1105000
26  2         44410023  59070019 14659997 *      2.001637    1.0434750
27  2         59070020  59089961    19942 *      2.166941    1.1219000
28  2         59089961  87059860 27969900 *      2.001637    1.0434750
29  2         88330099  89149998   819900 *      2.001637    1.0434750
30  2         89149998  89160000    10003 *      1.850960    0.9719899
⋮   ⋮        ⋮         ⋮         ⋮        ⋮      ⋮           ⋮        
295 X          1761586  58499983 56738398 *      1.003690231 1.0462   
296 X         61750000  79090023 17340024 *      0.995259022 1.0382   
297 X         79090023  79099982     9960 *      1.158297522 1.1929   
298 X         79099982  79270016   170035 *      0.954156879 0.9992   
299 X         79270016  79290103    20088 *      1.039522868 1.0802   
300 X         79290104  88459980  9169877 *      0.915794879 0.9628   
301 X         88990017  89349936   359920 *      0.855722516 0.9058   
302 X         90140199  91529952  1389754 *      0.866261527 0.9158   
303 X         92380000 105550099 13170100 *      0.977553484 1.0214   
304 X        105550100 105559976     9877 *      1.145334538 1.1806   
305 X        105559976 106940004  1380029 *      0.995996753 1.0389   
306 X        106940004 106949905     9902 *      1.210571016 1.2425   
307 X        106949905 134010283 27060379 *      1.004533352 1.0470   
308 X        134010284 134049988    39705 *      1.157665181 1.1923   
309 X        134049988 142369949  8319962 *      0.968911495 1.0132   
310 X        142369949 142380008    10060 *      1.124678077 1.1610   
311 X        142380008 151329236  8949229 *      0.977237313 1.0211   
312 X        151329237 151361865    32629 *      0.042216258 0.1339   
313 X        151361866 154929639  3567774 *      1.003584841 1.0461   
314 Y          2653090   2919837   266748 *      0.971335467 1.0155   
315 Y          3712028   3719919     7892 *      0.991570368 1.0347   
316 Y          6420014  10041053  3621040 *      0.904728918 0.9523   
317 Y         10041054  10069977    28924 *      0.018819654 0.1117   
318 Y         13130467  19558992  6428526 *      0.931498005 0.9777   
319 Y         20802956  2365

In [5]:
cnv <- fread("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/cnv/cnv.tsv")
dim(cnv)


[1] 1551825      15

In [6]:
rna_gt <- readRDS("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/drop_runs/drop_master_202502_allGenes/processed_data/mae/RNA_GT/DMENTM8.Rds")
rna_gt

GRanges object with 984 ranges and 1 metadata column:
        seqnames    ranges strand |      RNA_GT
           <Rle> <IRanges>  <Rle> | <character>
    [1]        1   1342612      * |         0/0
    [2]        1   1469546      * |         0/1
    [3]        1   2522204      * |         0/0
    [4]        1   3544187      * |         0/0
    [5]        1   6651093      * |         0/0
    ...      ...       ...    ... .         ...
  [980]        9 137734232      * |         0/1
  [981]        9 138396251      * |         0/1
  [982]        9 138831567      * |         0/0
  [983]        9 139566996      * |         0/0
  [984]        9 139732462      * |         0/0
  -------
  seqinfo: 22 sequences from an unspecified genome; no seqlengths

In [34]:
cnv_type <- as.data.table(assay(cnv, "Type"))
cnv_type[ WGS.MNL8UP.tumor == 1,]

WGS.MNL8UP.tumor
<chr>
1
1


In [10]:
assayNames(cnv_germline)

[1] "Type"       "CN"         "NP"         "QUAL"       "Confidence"
[6] "Gene"

In [ ]:
cdt <- as.data.table(colData(cnv))
colnames(cdt)
cdt[, c("SampleID", "VersionCNV", "")]

[1] "SampleID"                   "ProjectID"                 
 [3] "PatientID"                  "TumorID"                   
 [5] "ControlID"                  "DNASeq"                    
 [7] "RNASample"                  "NoDataReason"              
 [9] "DNASampleChar"              "Ploidy"                    
[11] "TumorCellContent"           "CNVSolutionConfidence"     
[13] "CNVSolutionMTB"             "DegradationControl"        
[15] "DegradationTumor"           "TelomereTiltControl"       
[17] "TelomereTiltTumor"          "SufficientTumorCellContent"
[19] "SevereDegradation"          "Comment"                   
[21] "DNASeqDateTumor"            "DNASeqDateControl"         
[23] "RNASeqDate"                 "DNAPlatformTumor"          
[25] "DNAPlatformControl"         "RNAPlatform"               
[27] "DNAPrepKitControl"          "DNAPrepKitTumor"           
[29] "DNAPrepKitCNV"              "RNAPrepKit"                
[31] "DNATumorID"                 "DNAControlID"              
[33] "RNASampleID"                "TissueMaterial"            
[35] "Sex"                        "VersionSNV"                
[37] "VersionInDel"               "VersionGermlineSmallVar"   
[39] "VersionGermlineCNV"         "VersionCNV"                
[41] "VersionSV"                  "VersionMSI"                
[43] "VersionRNA"                 "VersionFusion"             
[45] "LOH.HRD"                    "LST"                       
[47] "TAI"                        "MSI"                       
[49] "AdaptorHopping"             "INDEL_SomaticRescue"       
[51] "INDEL_SomaticPass"          "SNV_dbSNP"                 
[53] "SNV_TotalSomatic"           "INDEL_TotalSomatic"        
[55] "SNV_TotalGermline"          "INDEL_TotalGermline"       
[57] "SNV_ClinFunOnTarget"        "INDEL_ClinFunOnTarget"     
[59] "MutPerMb"

In [11]:
rdt <- as.data.table(rowData(cnv))
rdt

TCN,CovRatio,BAF,c1Mean,c2Mean,Type,LOH
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<lgl>
1.905110,0.9573419,NA,0.9525552,0.9525552,CNN,FALSE
2.197807,1.1044254,0.6166911,0.8424388,1.3553678,CNN,FALSE
1.974623,0.9922730,NA,NA,NA,CNN,FALSE
2.197097,1.1040687,0.6068063,0.8638846,1.3332122,CNN,FALSE
1.993741,1.0018799,NA,0.9968705,0.9968705,CNN,FALSE
1.914905,0.9622638,NA,0.9574525,0.9574525,CNN,FALSE
2.225633,1.1184083,NA,NA,NA,CNN,FALSE
1.980018,0.9949837,NA,0.9900088,0.9900088,CNN,FALSE
2.664904,1.3391476,NA,NA,NA,NA,FALSE
